# Agent 3 — Safety & Policy Guardrail

**Capstone build — Member C's deliverable**

This notebook builds **Agent 3** of the Automated Travel & Itinerary Coordinator: the travel advisor / risk assessor. It does three things:

| Job | Rubric link |
|---|---|
| Audits the itinerary against practical constraints (pacing, gaps, seasonal weather) | Multi-Agent System (D3) |
| Enforces the **PII output guardrail** — masks passport / phone / email | Security & Guardrails (D4) |
| Formats the `final_summary` for human review | feeds Member B's HITL node (D5) |

It reads from and writes to Member A's frozen `TravelState`, and runs **standalone** here against a mock itinerary — no need for Agents 1 or 2 to exist yet.

---
### Before running: add your API key
1. Click the **🔑 key icon** in the left Colab sidebar
2. Add secret: `GROQ_API_KEY` → your Groq key → toggle **ON**
---

In [1]:
# ============================================================
# STEP 1 — INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q langchain langgraph langchain-groq langchain-community

print("All packages installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All packages installed successfully.


In [11]:
# ============================================================
# STEP 2 — LOAD API KEY FROM COLAB SECRETS
# ============================================================

from google.colab import userdata
import os

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it in the Colab Secrets panel (🔑).")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("API key loaded successfully.")

API key loaded successfully.


In [3]:
# ============================================================
# STEP 3 — IMPORTS
# ============================================================

import re
import json
from datetime import datetime
from collections import defaultdict

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

print("Imports done.")

Imports done.


In [4]:
# ============================================================
# STEP 4 — SHARED STATE SCHEMA
# ============================================================

from typing import TypedDict, List, Dict, Optional, Literal, Annotated
import operator

class SearchResult(TypedDict):
    source: str
    query: str
    raw_content: str
    url: Optional[str]

class ReActStep(TypedDict):
    thought: str
    action: str
    action_input: str
    observation: str

class ItineraryItem(TypedDict):
    day: int
    activity: str
    location: str
    estimated_cost: float
    category: Literal["flight", "accommodation", "transit", "activity", "food", "entry_fee"]

class BudgetBreakdown(TypedDict):
    total_estimated_cost: float
    budget_limit: float
    over_budget: bool
    over_budget_amount: float
    cost_by_category: Dict[str, float]
    constraint_for_replanning: Optional[str]

class GuardrailLog(TypedDict):
    check_type: Literal["prompt_injection", "pii_masking"]
    triggered: bool
    details: str
    timestamp: str

class TravelState(TypedDict):
    user_request: str
    destination: str
    travel_dates: str
    budget_limit: float
    traveler_preferences: List[str]

    react_trace: Annotated[List[ReActStep], operator.add]
    search_results: Annotated[List[SearchResult], operator.add]
    draft_itinerary: List[ItineraryItem]

    budget_analysis: Optional[BudgetBreakdown]

    final_summary: Optional[str]
    audit_notes: List[str]

    guardrail_logs: Annotated[List[GuardrailLog], operator.add]
    pii_masked: bool

    iteration_count: int
    max_iterations: int
    replan_reason: Optional[str]

    human_approved: Optional[bool]
    human_feedback: Optional[str]

    execution_logs: Annotated[List[str], operator.add]
    status: Literal["in_progress", "awaiting_human", "completed", "failed"]

print("Shared state schema loaded.")

Shared state schema loaded.


In [12]:
# ============================================================
# STEP 5 — INITIALIZE LLM
# ============================================================

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("LLM ready.")

LLM ready.


In [6]:
# ============================================================
# STEP 6 — AGENT 3: SAFETY & POLICY GUARDRAIL AGENT
# ============================================================

class SafetyPolicyGuardrailAgent:

    # --- PII patterns -------------------------------------------------------
    EMAIL_RE    = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
    PASSPORT_RE = re.compile(r"\b[A-Z]{1,2}\d{6,9}\b")
    PHONE_RE    = re.compile(r"\+?\d{1,3}[\s-]?\d{2,4}[\s-]?\d{3}[\s-]?\d{3,4}")

    def __init__(self, llm):
        self.llm = llm

    # ---------- JOB 1a: deterministic rule-based audit ----------------------
    def rule_based_audit(self, itinerary):
        """
        Checks derived ONLY from fields that actually exist in ItineraryItem
        (day, category). No invented fields -> stays consistent with A's schema.
        """
        notes = []
        if not itinerary:
            return ["No itinerary items to audit."]

        per_day = defaultdict(list)
        for item in itinerary:
            per_day[item["day"]].append(item)

        # Over-packed day = proxy for 'missing rest interval'
        for day in sorted(per_day):
            activities = [i for i in per_day[day] if i["category"] == "activity"]
            if len(activities) > 4:
                notes.append(f"Day {day}: {len(activities)} activities — likely over-packed, no rest interval.")

        # Gap in the day numbering
        days = sorted(per_day)
        missing = [d for d in range(days[0], days[-1] + 1) if d not in per_day]
        if missing:
            notes.append(f"Itinerary has gaps: no plan for day(s) {missing}.")

        # Multi-day trip with no accommodation anywhere
        if len(days) > 1 and not any(i["category"] == "accommodation" for i in itinerary):
            notes.append("Multi-day trip but no accommodation item found.")

        return notes

    # ---------- JOB 1b: LLM-based judgment audit ----------------------------
    def llm_audit(self, state):
        """
        Judgment calls the rules can't make (seasonal/extreme weather, travel
        advisories). Instructor's JSON pattern: strict JSON out, strip fences,
        try/except with a safe fallback.
        """
        itinerary_text = "\n".join(
            f"Day {i['day']}: {i['activity']} @ {i['location']} ({i['category']})"
            for i in state["draft_itinerary"]
        )

        system_prompt = (
            "You are a travel safety advisor. Audit the itinerary below for "
            "PRACTICAL constraints only: extreme or seasonal weather for the given "
            "dates, known travel advisories, and pacing problems. "
            "Respond ONLY with a JSON object, no markdown fences:\n"
            '{ "concerns": ["short concern", "..."] }\n'
            "If there are no concerns, return an empty list.\n\n"
            f"Destination: {state['destination']}\n"
            f"Dates: {state['travel_dates']}\n"
            f"Itinerary:\n{itinerary_text}"
        )

        raw = self.llm.invoke([HumanMessage(content=system_prompt)]).content.strip()

        if raw.startswith("```"):
            raw = re.sub(r"^```[a-z]*\n?", "", raw)
            raw = re.sub(r"\n?```$", "", raw)

        try:
            data = json.loads(raw)
            return list(data.get("concerns", []))
        except Exception:
            return ["(LLM audit could not be parsed — manual review advised.)"]

    # ---------- JOB 2: PII output guardrail ---------------------------------
    def mask_pii(self, text):
        """
        Returns (masked_text, {type: count}).
        Order matters: email, then passport, then phone — so the specific
        patterns consume their characters before the looser phone pattern runs.
        """
        found = {}
        for pattern, label in [(self.EMAIL_RE, "EMAIL"),
                               (self.PASSPORT_RE, "PASSPORT"),
                               (self.PHONE_RE, "PHONE")]:
            matches = pattern.findall(text)
            if matches:
                found[label] = found.get(label, 0) + len(matches)
                text = pattern.sub(f"[REDACTED-{label}]", text)
        return text, found

    # ---------- JOB 3: format the human-review summary ----------------------
    def format_summary(self, state, audit_notes):
        """
        Builds the final_summary string. PII masking is applied to each
        free-text field HERE, before it enters the summary — never to the
        numeric cost fields, so real prices are untouched.
        """
        lines = [
            f"TRIP SUMMARY — {state['destination']}",
            f"Dates: {state['travel_dates']}",
            "",
            "Day-by-day plan:",
        ]

        pii_total = {}
        for item in sorted(state["draft_itinerary"], key=lambda x: x["day"]):
            activity, f1 = self.mask_pii(item["activity"])
            location, f2 = self.mask_pii(item["location"])
            for d in (f1, f2):
                for k, v in d.items():
                    pii_total[k] = pii_total.get(k, 0) + v
            lines.append(
                f"  Day {item['day']}: {activity} @ {location} "
                f"— {item['category']}, ~${item['estimated_cost']:.0f}"
            )

        lines.append("")

        budget = state.get("budget_analysis")
        if budget:
            status = ("within budget" if not budget["over_budget"]
                      else f"OVER by ${budget['over_budget_amount']:.0f}")
            lines += [
                "Budget:",
                f"  Estimated total: ${budget['total_estimated_cost']:.0f} "
                f"(limit ${budget['budget_limit']:.0f})",
                f"  Status: {status}",
                "",
            ]

        lines.append("Audit & safety notes:")
        if audit_notes:
            lines += [f"  - {n}" for n in audit_notes]
        else:
            lines.append("  - No practical concerns flagged.")

        lines += ["", "Ready for human review."]
        return "\n".join(lines), pii_total

    # ---------- Orchestrator: run all three jobs ----------------------------
    def run(self, state):
        audit_notes = self.rule_based_audit(state["draft_itinerary"]) + self.llm_audit(state)
        final_summary, pii_found = self.format_summary(state, audit_notes)
        return final_summary, audit_notes, pii_found


print("SafetyPolicyGuardrailAgent ready.")

SafetyPolicyGuardrailAgent ready.


In [7]:
# ============================================================
# STEP 7 — GRAPH NODE WRAPPER
# ============================================================


def agent3_audit_node(state: TravelState) -> dict:
    print("  [Agent 3 - Safety & Policy Guardrail] running.")

    agent = SafetyPolicyGuardrailAgent(llm)   # uses the shared global `llm`
    final_summary, audit_notes, pii_found = agent.run(state)

    guardrail_log = {
        "check_type": "pii_masking",
        "triggered": bool(pii_found),
        "details": (f"Masked PII: {pii_found}" if pii_found
                    else "No PII detected in itinerary text."),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    print(f"    audit notes: {len(audit_notes)}  |  PII masked: {pii_found or 'none'}")

    return {
        "final_summary": final_summary,
        "audit_notes":   audit_notes,
        "pii_masked":    True,
        "guardrail_logs": [guardrail_log],                                  # appended by reducer
        "execution_logs": ["Agent 3 (Safety & Policy Guardrail) executed"], # appended by reducer

        # COORDINATION POINT WITH MEMBER B:
        # We mark the state as ready for the human gate. If B prefers the HITL
        # node to own every status transition, just delete this one line.
        "status": "awaiting_human",
    }


print("agent3_audit_node ready — drops into the graph as 'agent3_audit'.")

agent3_audit_node ready — drops into the graph as 'agent3_audit'.


In [13]:
# ============================================================
# STEP 8 — STANDALONE SMOKE TEST (no other agents needed)
# ============================================================


mock_state: TravelState = {
    "user_request": "5-day cultural trip to Kyoto, mid-range budget",
    "destination": "Kyoto, Japan",
    "travel_dates": "2026-11-10 to 2026-11-15",
    "budget_limit": 2000.0,
    "traveler_preferences": ["temples", "food", "gardens"],

    "react_trace": [],
    "search_results": [],
    "draft_itinerary": [
        {"day": 1, "activity": "Check in (booking passport A1234567, contact guest@mail.com)",
         "location": "Gion", "estimated_cost": 120.0, "category": "accommodation"},
        {"day": 1, "activity": "Evening food tour", "location": "Pontocho",
         "estimated_cost": 60.0, "category": "food"},
        {"day": 2, "activity": "Fushimi Inari shrine", "location": "Fushimi",
         "estimated_cost": 0.0, "category": "activity"},
        {"day": 2, "activity": "Kiyomizu-dera", "location": "Higashiyama",
         "estimated_cost": 5.0, "category": "activity"},
        {"day": 2, "activity": "Nishiki market", "location": "Nakagyo",
         "estimated_cost": 30.0, "category": "activity"},
        {"day": 2, "activity": "Gion night walk", "location": "Gion",
         "estimated_cost": 0.0, "category": "activity"},
        {"day": 2, "activity": "Tea ceremony", "location": "Gion",
         "estimated_cost": 40.0, "category": "activity"},
        {"day": 4, "activity": "Arashiyama bamboo grove; guide +966 50 123 4567",
         "location": "Arashiyama", "estimated_cost": 25.0, "category": "activity"},
    ],

    "budget_analysis": {
        "total_estimated_cost": 1750.0,
        "budget_limit": 2000.0,
        "over_budget": False,
        "over_budget_amount": 0.0,
        "cost_by_category": {"accommodation": 600.0, "food": 400.0, "activity": 750.0},
        "constraint_for_replanning": None,
    },

    "final_summary": None,
    "audit_notes": [],
    "guardrail_logs": [],
    "pii_masked": False,
    "iteration_count": 1,
    "max_iterations": 3,
    "replan_reason": None,
    "human_approved": None,
    "human_feedback": None,
    "execution_logs": [],
    "status": "in_progress",
}

result = agent3_audit_node(mock_state)

print("\n================ FINAL SUMMARY ================")
print(result["final_summary"])

print("\n================ AUDIT NOTES =================")
for n in result["audit_notes"]:
    print(" -", n)

print("\n================ GUARDRAIL LOG ===============")
print(result["guardrail_logs"][0])

print("\nstatus ->", result["status"], "| pii_masked ->", result["pii_masked"])

  [Agent 3 - Safety & Policy Guardrail] running.
    audit notes: 5  |  PII masked: {'EMAIL': 1, 'PASSPORT': 1, 'PHONE': 1}

================ FINAL SUMMARY ================
TRIP SUMMARY — Kyoto, Japan
Dates: 2026-11-10 to 2026-11-15

Day-by-day plan:
  Day 1: Check in (booking passport [REDACTED-PASSPORT], contact [REDACTED-EMAIL]) @ Gion — accommodation, ~$120
  Day 1: Evening food tour @ Pontocho — food, ~$60
  Day 2: Fushimi Inari shrine @ Fushimi — activity, ~$0
  Day 2: Kiyomizu-dera @ Higashiyama — activity, ~$5
  Day 2: Nishiki market @ Nakagyo — activity, ~$30
  Day 2: Gion night walk @ Gion — activity, ~$0
  Day 2: Tea ceremony @ Gion — activity, ~$40
  Day 4: Arashiyama bamboo grove; guide [REDACTED-PHONE] @ Arashiyama — activity, ~$25

Budget:
  Estimated total: $1750 (limit $2000)
  Status: within budget

Audit & safety notes:
  - Day 2: 5 activities — likely over-packed, no rest interval.
  - Itinerary has gaps: no plan for day(s) [3].
  - Pacing problem on Day 2 with five

---
## What this notebook proves

| Component | Evidence when you run it |
|---|---|
| Rule-based audit | Flags Day-2 over-packing and the Day-3 gap |
| LLM judgment audit | Adds a seasonal/weather concern via Groq |
| **PII output guardrail** | Masks the planted passport, email, and phone in the summary, and logs it to `guardrail_logs` |
| Human-review summary | Produces the `final_summary` Member B's HITL node will show the human |

**Integration checklist (for when we merge into the main project notebook):**
- Delete **Step 4** here — the schema comes from Member A's Phase 0 cell.
- Keep only one shared `llm` (Step 5) — don't define a second.
- Register the node exactly as: `workflow.add_node("agent3_audit", agent3_audit_node)` (replaces A's stub).
- Confirm the `status = "awaiting_human"` hand-off convention with Member B.

**For D6 (evidence):** run all cells top-to-bottom, then save the notebook *with output* before committing — the rubric wants captured output, not just runnable code.